In [4]:
import os
import math
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import warnings
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from datetime import datetime

warnings.filterwarnings("ignore")

# Define the MyModel class to match the one used when saving the models
class MyModel(nn.Module):
    def __init__(self, input_dim=12):
        super(MyModel, self).__init__()
        self.layer1 = nn.Linear(input_dim, 450)
        self.bn1 = nn.BatchNorm1d(450)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(450, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

# =============================================================================
# KQ-related calculation functions
# =============================================================================

kq_element_properties = {
    'Nb': {
        'pauling_electronegativity': 1.60,
        'lattice_constant': 330.04,
        'melting_point_celsius': 2477,
        'C4 enthalpy melting': 26.8,
        'Electrophilicity Index': 1.261
    },
    'Si': {
        'pauling_electronegativity': 1.90,
        'lattice_constant': 543.09,
        'melting_point_celsius': 1414,
        'C4 enthalpy melting': 50.6,
        'Electrophilicity Index': 1.683
    },
    'Ti': {
        'pauling_electronegativity': 1.54,
        'lattice_constant': 295.08,
        'melting_point_celsius': 1668,
        'C4 enthalpy melting': 18.8,
        'Electrophilicity Index': 0.884
    },
    'Al': {
        'pauling_electronegativity': 1.61,
        'lattice_constant': 404.95,
        'melting_point_celsius': 660.32,
        'C4 enthalpy melting': 10.75,
        'Electrophilicity Index': 0.927
    },
    'Cr': {
        'pauling_electronegativity': 1.66,
        'lattice_constant': 291.00,
        'melting_point_celsius': 1907,
        'C4 enthalpy melting': 21.0,
        'Electrophilicity Index': 1.132
    },
    'Hf': {
        'pauling_electronegativity': 1.30,
        'lattice_constant': 319.64,
        'melting_point_celsius': 2223,
        'C4 enthalpy melting': 25.1,
        'Electrophilicity Index': 0.858
    },
    'Zr': {
        'pauling_electronegativity': 1.33,
        'lattice_constant': 323.20,
        'melting_point_celsius': 1855,
        'C4 enthalpy melting': 19.2,
        'Electrophilicity Index': 1.004
    },
    'V': {
        'pauling_electronegativity': 1.63,
        'lattice_constant': 303.00,
        'melting_point_celsius': 1910,
        'C4 enthalpy melting': 17.5,
        'Electrophilicity Index': 1.062
    }
}

kq_binary_enthalpy = {
    ('Nb', 'Si'): -56,  ('Nb', 'Ti'): 2,    ('Nb', 'Al'): -18, ('Nb', 'Cr'): -7,
    ('Nb', 'Hf'): 4,    ('Nb', 'Zr'): 4,    ('Nb', 'V'): -1,
    ('Si', 'Ti'): -66,  ('Si', 'Al'): -19,  ('Si', 'Cr'): -37, ('Si', 'Hf'): -77,
    ('Si', 'Zr'): -84,  ('Si', 'V'): -48,
    ('Ti', 'Al'): -30,  ('Ti', 'Cr'): -7,   ('Ti', 'Hf'):  0,  ('Ti', 'Zr'): 0,
    ('Ti', 'V'):  -2,
    ('Al', 'Cr'): -10,  ('Al', 'Hf'): -39,  ('Al', 'Zr'): -44, ('Al', 'V'): -16,
    ('Cr', 'Hf'): -9,   ('Cr', 'Zr'): -12,  ('Cr', 'V'): -2,
    ('Hf', 'Zr'): 0,    ('Hf', 'V'): -2,
    ('Zr', 'V'): -4
}

R = 8.314

def celsius_to_kelvin(t_celsius):
    return t_celsius + 273.15

def kq_calculate_x_and_delta_x(comp_dict):
    fractions = {el: pct/100.0 for el, pct in comp_dict.items()}
    x_sum = 0.0
    for el, frac in fractions.items():
        if el in kq_element_properties:
            x_sum += frac * kq_element_properties[el]['pauling_electronegativity']
    x_val = x_sum
    
    sum_sq = 0.0
    for el, frac in fractions.items():
        if el in kq_element_properties:
            diff = kq_element_properties[el]['pauling_electronegativity'] - x_val
            sum_sq += frac * (diff**2)
    delta_chi = math.sqrt(sum_sq)
    delta_x_val = delta_chi * 100
    return x_val, delta_x_val

def kq_calculate_mixing_entropy(comp_dict):
    fractions = [pct/100.0 for pct in comp_dict.values()]
    sum_tmp = 0.0
    for f in fractions:
        if f > 0:
            sum_tmp += f * math.log(f)
    return -R * sum_tmp

def kq_calculate_mixing_enthalpy(comp_dict):
    fractions = {el: pct/100.0 for el, pct in comp_dict.items()}
    el_list = list(fractions.keys())
    total_enthalpy = 0.0
    for i in range(len(el_list)):
        for j in range(i+1, len(el_list)):
            e1, e2 = el_list[i], el_list[j]
            if (e1, e2) in kq_binary_enthalpy:
                val = kq_binary_enthalpy[(e1, e2)]
            elif (e2, e1) in kq_binary_enthalpy:
                val = kq_binary_enthalpy[(e2, e1)]
            else:
                val = 0
            c1, c2 = fractions[e1], fractions[e2]
            total_enthalpy += 4 * val * c1 * c2
    return total_enthalpy

def kq_calculate_melting_point_in_K(comp_dict):
    fractions = {el: pct/100.0 for el, pct in comp_dict.items()}
    mp_c = 0.0
    for el, frac in fractions.items():
        if el in kq_element_properties:
            mp_c += frac * kq_element_properties[el]['melting_point_celsius']
    return celsius_to_kelvin(mp_c)

def kq_calculate_am(comp_dict):
    fractions = {el: pct/100.0 for el, pct in comp_dict.items()}
    am_val = 0.0
    for el, frac in fractions.items():
        if el in kq_element_properties:
            am_val += frac * kq_element_properties[el]['lattice_constant']
    return am_val

def kq_calculate_Omega(comp_dict):
    TmK = kq_calculate_melting_point_in_K(comp_dict)
    Smix = kq_calculate_mixing_entropy(comp_dict)
    Hmix = kq_calculate_mixing_enthalpy(comp_dict)
    if abs(Hmix) < 1e-12:
        return float('inf')
    else:
        return (TmK * Smix) / abs(Hmix * 1000.0)

def kq_calculate_mean_property(comp_dict, prop_key):
    fractions = {el: pct/100.0 for el, pct in comp_dict.items()}
    total = 0.0
    for el, frac in fractions.items():
        if el in kq_element_properties and prop_key in kq_element_properties[el]:
            total += frac * kq_element_properties[el][prop_key]
    return total

def compute_kq_features(composition_array, elements_order):
    comp_dict = dict(zip(elements_order, composition_array))
    _, delta_x_val = kq_calculate_x_and_delta_x(comp_dict)
    delta_h_mix = kq_calculate_mixing_enthalpy(comp_dict)
    am_val = kq_calculate_am(comp_dict)
    omega_val = kq_calculate_Omega(comp_dict)
    mean_c4 = kq_calculate_mean_property(comp_dict, 'C4 enthalpy melting')
    mean_ei = kq_calculate_mean_property(comp_dict, 'Electrophilicity Index')
    return [delta_x_val, delta_h_mix, am_val, omega_val, mean_c4, mean_ei]

# =============================================================================
# HTC-related calculation functions
# =============================================================================

htc_element_properties = {
    'Nb': {
        'valence_electrons': 5,
        'lattice_constant': 330.04,
        '热膨胀': 0.0000073,
        '比热容': 0.265,
        'Pyykko': 116,
        'Electron affinity': 86.1
    },
    'Si': {
        'valence_electrons': 4,
        'lattice_constant': 543.09,
        '热膨胀': 0.0000026,
        '比热容': 0.712,
        'Pyykko': 102,
        'Electron affinity': 133.6
    },
    'Ti': {
        'valence_electrons': 4,
        'lattice_constant': 295.08,
        '热膨胀': 0.0000086,
        '比热容': 0.523,
        'Pyykko': 108,
        'Electron affinity': 7.6
    },
    'Al': {
        'valence_electrons': 3,
        'lattice_constant': 404.95,
        '热膨胀': 0.0000231,
        '比热容': 0.897,
        'Pyykko': 111,
        'Electron affinity': 42.5
    },
    'Cr': {
        'valence_electrons': 6,
        'lattice_constant': 291,
        '热膨胀': 0.0000049,
        '比热容': 0.449,
        'Pyykko': 103,
        'Electron affinity': 64.3
    },
    'Hf': {
        'valence_electrons': 4,
        'lattice_constant': 319.64,
        '热膨胀': 0.0000059,
        '比热容': 0.144,
        'Pyykko': 122,
        'Electron affinity': 0
    },
    'Zr': {
        'valence_electrons': 4,
        'lattice_constant': 323.2,
        '热膨胀': 0.0000057,
        '比热容': 0.278,
        'Pyykko': 121,
        'Electron affinity': 41.1
    },
    'V': {
        'valence_electrons': 5,
        'lattice_constant': 303,
        '热膨胀': 0.0000084,
        '比热容': 0.489,
        'Pyykko': 106,
        'Electron affinity': 50.6
    }
}

def htc_calculate_mean_property(comp_dict, prop_key):
    fractions = {el: pct/100.0 for el, pct in comp_dict.items()}
    total = 0.0
    for el, frac in fractions.items():
        if el in htc_element_properties and prop_key in htc_element_properties[el]:
            total += frac * htc_element_properties[el][prop_key]
    return total

def htc_calculate_variance_property(comp_dict, prop_key):
    fractions = {el: pct/100.0 for el, pct in comp_dict.items()}
    mean_val = htc_calculate_mean_property(comp_dict, prop_key)
    var = 0.0
    for el, frac in fractions.items():
        if el in htc_element_properties and prop_key in htc_element_properties[el]:
            var += frac * (htc_element_properties[el][prop_key] - mean_val)**2
    return var

def htc_calculate_VEC(comp_dict):
    fractions = {el: pct/100.0 for el, pct in comp_dict.items()}
    vec = 0.0
    for el, frac in fractions.items():
        if el in htc_element_properties and 'valence_electrons' in htc_element_properties[el]:
            vec += frac * htc_element_properties[el]['valence_electrons']
    return vec

def compute_htc_features(composition_array, elements_order):
    comp_dict = dict(zip(elements_order, composition_array))

    vec_val = htc_calculate_VEC(comp_dict)
    am_val = htc_calculate_mean_property(comp_dict, 'lattice_constant')
    mean_thermal_expansion = htc_calculate_mean_property(comp_dict, '热膨胀')
    mean_specific_heat = htc_calculate_mean_property(comp_dict, '比热容')
    mean_pyykko = htc_calculate_mean_property(comp_dict, 'Pyykko')
    var_electron_affinity = htc_calculate_variance_property(comp_dict, 'Electron affinity')

    return [
        vec_val,
        am_val,
        mean_thermal_expansion,
        mean_specific_heat,
        mean_pyykko,
        var_electron_affinity
    ]
# =============================================================================
# NSGA-II Algorithm
# =============================================================================

class CustomNSGA2:
    def __init__(self, 
                 kq_model, 
                 kq_scaler, 
                 htc_model, 
                 htc_scaler, 
                 elements_list,
                 pop_size=500, 
                 max_generations=250,
                 min_generations=20,
                 kq_weight=0.75,
                 htc_weight=0.25,
                 convergence_window=5,
                 hv_epsilon=0.001,
                 score_epsilon=0.0005,
                 min_htc_threshold=30,
                 random_seed=None):
        
        self.kq_model = kq_model
        self.kq_scaler = kq_scaler
        self.htc_model = htc_model
        self.htc_scaler = htc_scaler
        self.pop_size = pop_size
        
        self.max_generations = max_generations
        self.min_generations = min_generations
        self.kq_weight = kq_weight
        self.htc_weight = htc_weight
        self.convergence_window = convergence_window
        self.hv_epsilon = hv_epsilon
        self.score_epsilon = score_epsilon
        self.min_htc_threshold = min_htc_threshold
        self.random_seed = random_seed
        
        if random_seed is not None:
            random.seed(random_seed)
            np.random.seed(random_seed)
            torch.manual_seed(random_seed)
        
        self.elements_order = elements_list
        self.variable_elements = elements_list[1:]
        self.num_vars = len(self.variable_elements)
        
        self.lower_bounds = np.array([12 if el == 'Si' else 20 if el == 'Ti' else 0 for el in self.variable_elements])
        self.upper_bounds = np.array([18 if el == 'Si' else 35 if el == 'Ti' else 5 for el in self.variable_elements])
        
        self.convergence_history = {
            'generation': [],
            'best_kq': [],
            'best_htc': [],
            'best_combined_score': [],
            'mean_kq': [],
            'mean_htc': [],
            'pareto_count': [],
            'evaluations': []
        }
        self.total_evaluations = 0
    
    def initialize_population(self):
        population = []
        for _ in range(self.pop_size):
            individual = np.zeros(self.num_vars)
            for i in range(self.num_vars):
                individual[i] = random.uniform(self.lower_bounds[i], self.upper_bounds[i])
            
            var_sum = np.sum(individual)
            nb = 100 - var_sum
            
            if 35 <= nb <= 55:
                population.append(individual)
        
        while len(population) < self.pop_size:
            individual = np.zeros(self.num_vars)
            for i in range(self.num_vars):
                individual[i] = random.uniform(self.lower_bounds[i], self.upper_bounds[i])
            
            var_sum = np.sum(individual)
            nb = 100 - var_sum
            
            if 35 <= nb <= 55:
                population.append(individual)
        
        return np.array(population)
    
    def evaluate_individual(self, individual):
        self.total_evaluations += 1
        
        var_sum = np.sum(individual)
        nb = 100 - var_sum
        
        if not (35 <= nb <= 55):
            return float('-inf'), float('-inf')
        
        composition = [nb] + list(individual)
        full_composition = composition + [0] * (8 - len(composition))
        
        try:
            kq_feats = compute_kq_features(full_composition, ['Nb', 'Si', 'Ti', 'Al', 'Cr', 'Hf', 'Zr', 'V'])
            kq_feats_scaled = self.kq_scaler.transform([kq_feats])
            kq_input = torch.from_numpy(kq_feats_scaled.astype(np.float32))
            with torch.no_grad():
                kq_pred = self.kq_model(kq_input).cpu().numpy().flatten()[0]
            
            htc_feats = compute_htc_features(full_composition, ['Nb', 'Si', 'Ti', 'Al', 'Cr', 'Hf', 'Zr', 'V'])
            htc_feats_scaled = self.htc_scaler.transform([htc_feats])
            htc_input = torch.from_numpy(htc_feats_scaled.astype(np.float32))
            with torch.no_grad():
                htc_pred = self.htc_model(htc_input).cpu().numpy().flatten()[0]
            
            if htc_pred < self.min_htc_threshold:
                return float('-inf'), float('-inf')
                
            return kq_pred, htc_pred
        except Exception as e:
            print(f"评估成分时出错: {e}")
            return float('-inf'), float('-inf')
    
    def evaluate_population(self, population):
        kq_vals, htc_vals = [], []
        for ind in population:
            kq, htc = self.evaluate_individual(ind)
            kq_vals.append(kq)
            htc_vals.append(htc)
        return np.array(kq_vals), np.array(htc_vals)
    
    def is_dominating(self, ind1, ind2, fitness1, fitness2):
        kq1, htc1 = fitness1
        kq2, htc2 = fitness2
        return (kq1 >= kq2 and htc1 > htc2) or (kq1 > kq2 and htc1 >= htc2)
    
    def fast_non_dominated_sort(self, population, kq_values, htc_values):
        n = len(population)
        domination_counts = np.zeros(n, dtype=int)
        dominated_solutions = [[] for _ in range(n)]
        fronts = [[]]
        for i in range(n):
            for j in range(n):
                if i != j:
                    if self.is_dominating(population[i], population[j],
                                          (kq_values[i], htc_values[i]),
                                          (kq_values[j], htc_values[j])):
                        dominated_solutions[i].append(j)
                    elif self.is_dominating(population[j], population[i],
                                            (kq_values[j], htc_values[j]),
                                            (kq_values[i], htc_values[i])):
                        domination_counts[i] += 1
            if domination_counts[i] == 0:
                fronts[0].append(i)
        k = 0
        while fronts[k]:
            next_front = []
            for i in fronts[k]:
                for j in dominated_solutions[i]:
                    domination_counts[j] -= 1
                    if domination_counts[j] == 0:
                        next_front.append(j)
            k += 1
            fronts.append(next_front)
        fronts.pop()
        return fronts
    
    def calculate_crowding_distance(self, population, kq_values, htc_values, front):
        if len(front) <= 2:
            return [float('inf')] * len(front)
        n = len(front)
        distances = [0.0] * n
        for values in [kq_values, htc_values]:
            idx_sorted = sorted(range(n), key=lambda i: values[front[i]])
            distances[idx_sorted[0]] = float('inf')
            distances[idx_sorted[-1]] = float('inf')
            obj_range = values[front[idx_sorted[-1]]] - values[front[idx_sorted[0]]]
            if obj_range > 0:
                for i in range(1, n-1):
                    distances[idx_sorted[i]] += (values[front[idx_sorted[i+1]]] 
                                                 - values[front[idx_sorted[i-1]]]) / obj_range
        return distances
    
    def crossover(self, parent1, parent2):
        eta = 15
        child1 = np.zeros_like(parent1)
        child2 = np.zeros_like(parent2)
        for i in range(len(parent1)):
            if random.random() > 0.9:
                child1[i], child2[i] = parent1[i], parent2[i]
                continue
            if abs(parent1[i] - parent2[i]) > 1e-10:
                if parent1[i] < parent2[i]:
                    y1, y2 = parent1[i], parent2[i]
                else:
                    y1, y2 = parent2[i], parent1[i]
                lb, ub = self.lower_bounds[i], self.upper_bounds[i]
                beta = 1.0 + (2.0 * (y1 - lb) / (y2 - y1))
                alpha = 2.0 - beta ** (-(eta + 1))
                u = random.random()
                if u <= 1.0 / alpha:
                    beta_q = (u * alpha) ** (1.0 / (eta + 1))
                else:
                    beta_q = (1.0 / (2.0 - u * alpha)) ** (1.0 / (eta + 1))
                c1 = 0.5 * ((y1 + y2) - beta_q * (y2 - y1))
                c2 = 0.5 * ((y1 + y2) + beta_q * (y2 - y1))
                c1 = min(max(c1, lb), ub)
                c2 = min(max(c2, lb), ub)
                if random.random() <= 0.5:
                    child1[i], child2[i] = c1, c2
                else:
                    child1[i], child2[i] = c2, c1
            else:
                child1[i], child2[i] = parent1[i], parent2[i]
        return child1, child2
    
    def mutation(self, individual):
        eta = 20
        child = individual.copy()
        for i in range(len(child)):
            if random.random() > 0.1:
                continue
            lb, ub = self.lower_bounds[i], self.upper_bounds[i]
            delta1 = (child[i] - lb) / (ub - lb)
            delta2 = (ub - child[i]) / (ub - lb)
            mut_pow = 1.0 / (eta + 1.0)
            r = random.random()
            if r <= 0.5:
                xy = 1.0 - delta1
                val = 2.0 * r + (1.0 - 2.0 * r) * (xy ** (eta + 1.0))
                delta_q = val ** mut_pow - 1.0
            else:
                xy = 1.0 - delta2
                val = 2.0 * (1.0 - r) + 2.0 * (r - 0.5) * (xy ** (eta + 1.0))
                delta_q = 1.0 - (val ** mut_pow)
            child[i] += delta_q * (ub - lb)
            child[i] = min(max(child[i], lb), ub)
        return child
    
    def tournament_selection(self, population, fronts, crowding_distances):
        selected = []
        for _ in range(len(population)):
            a, b = random.sample(range(len(population)), 2)
            a_front = next((i for i, front in enumerate(fronts) if a in front), float('inf'))
            b_front = next((i for i, front in enumerate(fronts) if b in front), float('inf'))
            if a_front < len(fronts):
                a_idx = fronts[a_front].index(a)
                a_distance = crowding_distances[a_front][a_idx]
            else:
                a_distance = -float('inf')
            if b_front < len(fronts):
                b_idx = fronts[b_front].index(b)
                b_distance = crowding_distances[b_front][b_idx]
            else:
                b_distance = -float('inf')
            if a_front < b_front:
                selected.append(population[a].copy())
            elif b_front < a_front:
                selected.append(population[b].copy())
            elif a_distance > b_distance:
                selected.append(population[a].copy())
            else:
                selected.append(population[b].copy())
        return np.array(selected)
    
    def repair_individual(self, individual):
        var_sum = np.sum(individual)
        nb = 100 - var_sum
        
        if 35 <= nb <= 55:
            return individual
        
        if nb < 35:
            deficit = 35 - nb
            total_others = var_sum
            scale_factor = (total_others - deficit) / total_others
            new_individual = individual * scale_factor
            
            for i in range(len(new_individual)):
                if new_individual[i] < self.lower_bounds[i]:
                    new_individual[i] = self.lower_bounds[i]
            
            new_var_sum = np.sum(new_individual)
            new_nb = 100 - new_var_sum
            
            if new_nb < 35:
                diff = 35 - new_nb
                si_idx = 0
                if new_individual[si_idx] - diff >= self.lower_bounds[si_idx]:
                    new_individual[si_idx] -= diff
            
            return new_individual
        
        elif nb > 55:
            excess = nb - 55
            total_others = var_sum
            scale_factor = (total_others + excess) / total_others
            new_individual = individual * scale_factor
            
            for i in range(len(new_individual)):
                if new_individual[i] > self.upper_bounds[i]:
                    new_individual[i] = self.upper_bounds[i]
            
            new_var_sum = np.sum(new_individual)
            new_nb = 100 - new_var_sum
            
            if new_nb > 55:
                diff = new_nb - 55
                si_idx = 0
                if new_individual[si_idx] + diff <= self.upper_bounds[si_idx]:
                    new_individual[si_idx] += diff
            
            return new_individual
        
        return individual
    
    def run(self, verbose=True):
        if verbose:
            print(f"\n运行NSGA-II优化 ({', '.join(self.elements_order)})...")
            print(f"种群大小: {self.pop_size}, 最大代数: {self.max_generations}")
        
        start_time = time.time()
        population = self.initialize_population()
        
        all_solutions = []
        
        for gen in range(self.max_generations):
            if verbose and (gen + 1) % 50 == 0:
                print(f"代数 {gen+1}/{self.max_generations}")
            
            kq_values, htc_values = self.evaluate_population(population)
            
            valid_indices = [(i, kq, htc) for i, (kq, htc) in enumerate(zip(kq_values, htc_values)) 
                             if kq != float('-inf') and htc != float('-inf')]
            
            if not valid_indices:
                if verbose:
                    print("本代中没有有效的解决方案，重新生成种群...")
                population = self.initialize_population()
                continue
            
            valid_population = np.array([population[i] for i, _, _ in valid_indices])
            valid_kq = np.array([kq for _, kq, _ in valid_indices])
            valid_htc = np.array([htc for _, _, htc in valid_indices])
            
            if len(valid_population) < 2:
                if verbose:
                    print("有效种群数量不足，重新生成种群...")
                population = self.initialize_population()
                continue
            
            fronts = self.fast_non_dominated_sort(valid_population, valid_kq, valid_htc)
            
            crowding_distances = []
            for front in fronts:
                distances = self.calculate_crowding_distance(valid_population, valid_kq, valid_htc, front)
                crowding_distances.append(distances)
            
            self.convergence_history['generation'].append(gen + 1)
            self.convergence_history['best_kq'].append(valid_kq.max())
            self.convergence_history['best_htc'].append(valid_htc.max())
            self.convergence_history['mean_kq'].append(valid_kq.mean())
            self.convergence_history['mean_htc'].append(valid_htc.mean())
            self.convergence_history['pareto_count'].append(len(fronts[0]) if fronts else 0)
            self.convergence_history['evaluations'].append(self.total_evaluations)
            
            if len(valid_kq) > 0:
                min_kq = valid_kq.min()
                max_kq = valid_kq.max()
                min_htc = valid_htc.min()
                max_htc = valid_htc.max()
                
                if max_kq > min_kq:
                    norm_kq = (valid_kq - min_kq) / (max_kq - min_kq)
                else:
                    norm_kq = np.ones_like(valid_kq) * 0.5
                    
                if max_htc > min_htc:
                    norm_htc = (valid_htc - min_htc) / (max_htc - min_htc)
                else:
                    norm_htc = np.ones_like(valid_htc) * 0.5
                
                combined_scores = self.kq_weight * norm_kq + self.htc_weight * norm_htc
                self.convergence_history['best_combined_score'].append(combined_scores.max())
            else:
                self.convergence_history['best_combined_score'].append(0)
            
            for i, ind in enumerate(valid_population):
                var_sum = np.sum(ind)
                nb = 100 - var_sum
                
                composition = [nb] + list(ind)
                full_composition = composition + [0] * (8 - len(composition))
                
                solution = {}
                for j, element in enumerate(self.elements_order):
                    if j < len(composition):
                        solution[element] = round(composition[j], 2)
                    else:
                        solution[element] = 0
                
                solution['Predicted_KQ'] = valid_kq[i]
                solution['Predicted_HTC'] = valid_htc[i]
                solution['Generation'] = gen + 1
                solution['Is_Pareto'] = i in fronts[0] if fronts else False
                
                all_solutions.append(solution)
            
            selected = self.tournament_selection(valid_population, fronts, crowding_distances)
            
            if len(selected) < 2:
                if verbose:
                    print("选择的个体数量不足，重新生成种群...")
                population = self.initialize_population()
                continue
            
            offspring = []
            for i in range(0, len(selected), 2):
                if i + 1 < len(selected):
                    parent1 = selected[i]
                    parent2 = selected[i+1]
                    child1, child2 = self.crossover(parent1, parent2)
                    child1 = self.mutation(child1)
                    child2 = self.mutation(child2)
                    child1 = self.repair_individual(child1)
                    child2 = self.repair_individual(child2)
                    offspring.append(child1)
                    offspring.append(child2)
            
            if len(offspring) < self.pop_size:
                while len(offspring) < self.pop_size:
                    extra_parent = selected[random.randint(0, len(selected)-1)]
                    extra_child = self.mutation(extra_parent.copy())
                    extra_child = self.repair_individual(extra_child)
                    offspring.append(extra_child)
            
            population = np.array(offspring)
        
        elapsed_time = time.time() - start_time
        
        if all_solutions:
            df_results = pd.DataFrame(all_solutions)
            
            min_kq = df_results['Predicted_KQ'].min()
            max_kq = df_results['Predicted_KQ'].max()
            min_htc = df_results['Predicted_HTC'].min()
            max_htc = df_results['Predicted_HTC'].max()
            
            if max_kq > min_kq:
                df_results['Norm_KQ'] = (df_results['Predicted_KQ'] - min_kq) / (max_kq - min_kq)
            else:
                df_results['Norm_KQ'] = 0.5
                
            if max_htc > min_htc:
                df_results['Norm_HTC'] = (df_results['Predicted_HTC'] - min_htc) / (max_htc - min_htc)
            else:
                df_results['Norm_HTC'] = 0.5
            
            df_results['Combined_Score'] = self.kq_weight * df_results['Norm_KQ'] + self.htc_weight * df_results['Norm_HTC']
            
            df_results.sort_values(by='Combined_Score', ascending=False, inplace=True)
            df_results.reset_index(drop=True, inplace=True)
            
            return df_results, elapsed_time, self.convergence_history
        else:
            if verbose:
                print("没有找到有效解决方案!")
            return None, elapsed_time, self.convergence_history

# =============================================================================
# Random Search Algorithm
# =============================================================================

class RandomSearch:
    def __init__(self, 
                 kq_model, 
                 kq_scaler, 
                 htc_model, 
                 htc_scaler, 
                 elements_list,
                 n_samples=10000,
                 kq_weight=0.75,
                 htc_weight=0.25,
                 min_htc_threshold=30,
                 random_seed=None):
        
        self.kq_model = kq_model
        self.kq_scaler = kq_scaler
        self.htc_model = htc_model
        self.htc_scaler = htc_scaler
        self.n_samples = n_samples
        self.kq_weight = kq_weight
        self.htc_weight = htc_weight
        self.min_htc_threshold = min_htc_threshold
        self.random_seed = random_seed
        
        if random_seed is not None:
            random.seed(random_seed)
            np.random.seed(random_seed)
            torch.manual_seed(random_seed)
        
        self.elements_order = elements_list
        self.variable_elements = elements_list[1:]
        self.num_vars = len(self.variable_elements)
        
        self.lower_bounds = np.array([12 if el == 'Si' else 20 if el == 'Ti' else 0 for el in self.variable_elements])
        self.upper_bounds = np.array([18 if el == 'Si' else 35 if el == 'Ti' else 5 for el in self.variable_elements])
        
        self.convergence_history = {
            'sample': [],
            'best_kq': [],
            'best_htc': [],
            'best_combined_score': [],
            'constraint_satisfied': [],
            'evaluations': []
        }
        self.total_evaluations = 0
    
    def generate_random_individual(self):
        individual = np.zeros(self.num_vars)
        for i in range(self.num_vars):
            individual[i] = random.uniform(self.lower_bounds[i], self.upper_bounds[i])
        return individual
    
    def evaluate_individual(self, individual):
        self.total_evaluations += 1
        
        var_sum = np.sum(individual)
        nb = 100 - var_sum
        
        if not (35 <= nb <= 55):
            return float('-inf'), float('-inf'), False
        
        composition = [nb] + list(individual)
        full_composition = composition + [0] * (8 - len(composition))
        
        try:
            kq_feats = compute_kq_features(full_composition, ['Nb', 'Si', 'Ti', 'Al', 'Cr', 'Hf', 'Zr', 'V'])
            kq_feats_scaled = self.kq_scaler.transform([kq_feats])
            kq_input = torch.from_numpy(kq_feats_scaled.astype(np.float32))
            with torch.no_grad():
                kq_pred = self.kq_model(kq_input).cpu().numpy().flatten()[0]
            
            htc_feats = compute_htc_features(full_composition, ['Nb', 'Si', 'Ti', 'Al', 'Cr', 'Hf', 'Zr', 'V'])
            htc_feats_scaled = self.htc_scaler.transform([htc_feats])
            htc_input = torch.from_numpy(htc_feats_scaled.astype(np.float32))
            with torch.no_grad():
                htc_pred = self.htc_model(htc_input).cpu().numpy().flatten()[0]
            
            if htc_pred < self.min_htc_threshold:
                return float('-inf'), float('-inf'), True
                
            return kq_pred, htc_pred, True
        except Exception as e:
            print(f"评估成分时出错: {e}")
            return float('-inf'), float('-inf'), False
    
    def run(self, verbose=True):
        if verbose:
            print(f"\n运行随机搜索优化 ({', '.join(self.elements_order)})...")
            print(f"采样次数: {self.n_samples}")
        
        start_time = time.time()
        
        all_solutions = []
        best_kq = float('-inf')
        best_htc = float('-inf')
        best_combined = float('-inf')
        
        constraint_satisfied_count = 0
        valid_solutions_count = 0
        
        for i in range(self.n_samples):
            if verbose and (i + 1) % 2000 == 0:
                print(f"采样 {i+1}/{self.n_samples}")
            
            individual = self.generate_random_individual()
            kq_val, htc_val, constraint_ok = self.evaluate_individual(individual)
            
            if constraint_ok:
                constraint_satisfied_count += 1
            
            if kq_val != float('-inf') and htc_val != float('-inf'):
                valid_solutions_count += 1
                
                var_sum = np.sum(individual)
                nb = 100 - var_sum
                
                composition = [nb] + list(individual)
                full_composition = composition + [0] * (8 - len(composition))
                
                solution = {}
                for j, element in enumerate(self.elements_order):
                    if j < len(composition):
                        solution[element] = round(composition[j], 2)
                    else:
                        solution[element] = 0
                
                solution['Predicted_KQ'] = kq_val
                solution['Predicted_HTC'] = htc_val
                solution['Sample'] = i + 1
                
                all_solutions.append(solution)
                
                if kq_val > best_kq:
                    best_kq = kq_val
                if htc_val > best_htc:
                    best_htc = htc_val
            
            if (i + 1) % 100 == 0:
                self.convergence_history['sample'].append(i + 1)
                self.convergence_history['best_kq'].append(best_kq)
                self.convergence_history['best_htc'].append(best_htc)
                self.convergence_history['constraint_satisfied'].append(
                    constraint_satisfied_count / (i + 1) * 100
                )
                self.convergence_history['evaluations'].append(self.total_evaluations)
                
                if len(all_solutions) > 0:
                    temp_df = pd.DataFrame(all_solutions)
                    min_kq = temp_df['Predicted_KQ'].min()
                    max_kq = temp_df['Predicted_KQ'].max()
                    min_htc = temp_df['Predicted_HTC'].min()
                    max_htc = temp_df['Predicted_HTC'].max()
                    
                    if max_kq > min_kq:
                        norm_kq = (temp_df['Predicted_KQ'] - min_kq) / (max_kq - min_kq)
                    else:
                        norm_kq = 0.5
                    
                    if max_htc > min_htc:
                        norm_htc = (temp_df['Predicted_HTC'] - min_htc) / (max_htc - min_htc)
                    else:
                        norm_htc = 0.5
                    
                    combined = self.kq_weight * norm_kq + self.htc_weight * norm_htc
                    best_combined = combined.max()
                
                self.convergence_history['best_combined_score'].append(best_combined)
        
        elapsed_time = time.time() - start_time
        
        constraint_satisfaction_rate = constraint_satisfied_count / self.n_samples * 100
        
        if all_solutions:
            df_results = pd.DataFrame(all_solutions)
            
            min_kq = df_results['Predicted_KQ'].min()
            max_kq = df_results['Predicted_KQ'].max()
            min_htc = df_results['Predicted_HTC'].min()
            max_htc = df_results['Predicted_HTC'].max()
            
            if max_kq > min_kq:
                df_results['Norm_KQ'] = (df_results['Predicted_KQ'] - min_kq) / (max_kq - min_kq)
            else:
                df_results['Norm_KQ'] = 0.5
                
            if max_htc > min_htc:
                df_results['Norm_HTC'] = (df_results['Predicted_HTC'] - min_htc) / (max_htc - min_htc)
            else:
                df_results['Norm_HTC'] = 0.5
            
            df_results['Combined_Score'] = self.kq_weight * df_results['Norm_KQ'] + self.htc_weight * df_results['Norm_HTC']
            
            df_results.sort_values(by='Combined_Score', ascending=False, inplace=True)
            df_results.reset_index(drop=True, inplace=True)
            
            return df_results, elapsed_time, constraint_satisfaction_rate, self.convergence_history
        else:
            if verbose:
                print("没有找到有效解决方案!")
            return None, elapsed_time, constraint_satisfaction_rate, self.convergence_history
# =============================================================================
# Comparison Functions
# =============================================================================

def compare_methods(nsga2_results, nsga2_time, nsga2_history, nsga2_total_evals,
                   random_results, random_time, random_constraint_rate, random_history, random_total_evals,
                   elements_list, output_dir):
    
    print("\n========= 开始对比分析 =========")
    
    elements_str = ''.join([el[0] for el in elements_list])
    comparison_dir = os.path.join(output_dir, f"comparison_{len(elements_list)}元_{elements_str}")
    if not os.path.exists(comparison_dir):
        os.makedirs(comparison_dir)
    
    plots_dir = os.path.join(comparison_dir, "plots")
    if not os.path.exists(plots_dir):
        os.makedirs(plots_dir)
    
    excel_dir = os.path.join(comparison_dir, "excel_data")
    if not os.path.exists(excel_dir):
        os.makedirs(excel_dir)
    
    comparison_stats = {}
    
    nsga2_pareto = nsga2_results[nsga2_results['Is_Pareto'] == True]
    comparison_stats['Method'] = ['NSGA-II', 'Random Search']
    comparison_stats['Total_Evaluations'] = [nsga2_total_evals, random_total_evals]
    comparison_stats['Time_seconds'] = [nsga2_time, random_time]
    comparison_stats['Valid_Solutions'] = [len(nsga2_results), len(random_results)]
    comparison_stats['Pareto_Solutions'] = [len(nsga2_pareto), 0]
    comparison_stats['Best_KQ'] = [nsga2_results['Predicted_KQ'].max(), random_results['Predicted_KQ'].max()]
    comparison_stats['Best_HTC'] = [nsga2_results['Predicted_HTC'].max(), random_results['Predicted_HTC'].max()]
    comparison_stats['Mean_KQ'] = [nsga2_results['Predicted_KQ'].mean(), random_results['Predicted_KQ'].mean()]
    comparison_stats['Mean_HTC'] = [nsga2_results['Predicted_HTC'].mean(), random_results['Predicted_HTC'].mean()]
    comparison_stats['Std_KQ'] = [nsga2_results['Predicted_KQ'].std(), random_results['Predicted_KQ'].std()]
    comparison_stats['Std_HTC'] = [nsga2_results['Predicted_HTC'].std(), random_results['Predicted_HTC'].std()]
    comparison_stats['Best_Combined_Score'] = [nsga2_results['Combined_Score'].max(), random_results['Combined_Score'].max()]
    
    nsga2_constraint_rate = 100.0
    comparison_stats['Constraint_Satisfaction_Rate_%'] = [nsga2_constraint_rate, random_constraint_rate]
    
    comparison_stats['Efficiency_BestKQ_per_Second'] = [
        nsga2_results['Predicted_KQ'].max() / nsga2_time,
        random_results['Predicted_KQ'].max() / random_time
    ]
    
    comparison_stats['Efficiency_BestCombined_per_1000Evals'] = [
        nsga2_results['Combined_Score'].max() / (nsga2_total_evals / 1000),
        random_results['Combined_Score'].max() / (random_total_evals / 1000)
    ]
    
    stats_df = pd.DataFrame(comparison_stats)
    
    stats_file = os.path.join(excel_dir, 'comparison_statistics.xlsx')
    with pd.ExcelWriter(stats_file, engine='openpyxl') as writer:
        stats_df.to_excel(writer, sheet_name='Statistics', index=False)
    
    print(f"已保存统计对比数据: {stats_file}")
    
    pareto_file = os.path.join(excel_dir, 'nsga2_pareto_solutions.xlsx')
    nsga2_pareto.to_excel(pareto_file, index=False)
    print(f"已保存NSGA-II Pareto解: {pareto_file}")
    
    random_top_file = os.path.join(excel_dir, 'random_search_top100_solutions.xlsx')
    random_results.head(100).to_excel(random_top_file, index=False)
    print(f"已保存随机搜索前100解: {random_top_file}")
    
    convergence_file = os.path.join(excel_dir, 'convergence_data.xlsx')
    with pd.ExcelWriter(convergence_file, engine='openpyxl') as writer:
        nsga2_conv_df = pd.DataFrame(nsga2_history)
        nsga2_conv_df.to_excel(writer, sheet_name='NSGA2_Convergence', index=False)
        
        random_conv_df = pd.DataFrame(random_history)
        random_conv_df.to_excel(writer, sheet_name='Random_Convergence', index=False)
    
    print(f"已保存收敛数据: {convergence_file}")
    
    create_comparison_plots(nsga2_results, nsga2_pareto, nsga2_history,
                           random_results, random_history,
                           stats_df, plots_dir, excel_dir, elements_list)
    
    return stats_df, comparison_dir

def create_comparison_plots(nsga2_results, nsga2_pareto, nsga2_history,
                           random_results, random_history,
                           stats_df, plots_dir, excel_dir, elements_list):
    
    elements_str = ''.join([el[0] for el in elements_list])
    
    print("\n创建对比可视化图表...")
    
    plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    
    # =========================================================================
    # 图1: Pareto前沿对比 (保持不变)
    # =========================================================================
    plt.figure(figsize=(12, 10))
    
    plt.scatter(random_results['Predicted_KQ'], random_results['Predicted_HTC'], 
               s=20, alpha=0.3, c='gray', label='Random Search (All)', marker='o')
    
    random_top10 = random_results.head(10)
    plt.scatter(random_top10['Predicted_KQ'], random_top10['Predicted_HTC'], 
               s=100, alpha=0.8, c='orange', label='Random Search (Top 10)', marker='*')
    
    plt.scatter(nsga2_results['Predicted_KQ'], nsga2_results['Predicted_HTC'], 
               s=20, alpha=0.5, c='lightblue', label='NSGA-II (All)', marker='o')
    
    plt.scatter(nsga2_pareto['Predicted_KQ'], nsga2_pareto['Predicted_HTC'], 
               s=80, alpha=0.9, c='blue', label='NSGA-II (Pareto Front)', marker='D', edgecolors='black')
    
    plt.xlabel('KQ (MPa·m^1/2)', fontsize=14, fontweight='bold')
    plt.ylabel('High Temperature Compression (MPa)', fontsize=14, fontweight='bold')
    plt.title(f'{len(elements_list)}-Element Alloy: Pareto Front Comparison', fontsize=16, fontweight='bold')
    plt.legend(fontsize=12, loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    plot1_file = os.path.join(plots_dir, f'1_pareto_front_comparison_{elements_str}.png')
    plt.savefig(plot1_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"已保存图1: Pareto前沿对比")
    
    pareto_comparison_data = pd.DataFrame({
        'Method': ['Random_All'] * len(random_results) + ['Random_Top10'] * len(random_top10) + 
                  ['NSGA2_All'] * len(nsga2_results) + ['NSGA2_Pareto'] * len(nsga2_pareto),
        'KQ': list(random_results['Predicted_KQ']) + list(random_top10['Predicted_KQ']) + 
              list(nsga2_results['Predicted_KQ']) + list(nsga2_pareto['Predicted_KQ']),
        'HTC': list(random_results['Predicted_HTC']) + list(random_top10['Predicted_HTC']) + 
               list(nsga2_results['Predicted_HTC']) + list(nsga2_pareto['Predicted_HTC'])
    })
    pareto_data_file = os.path.join(excel_dir, 'plot1_pareto_front_comparison_data.xlsx')
    pareto_comparison_data.to_excel(pareto_data_file, index=False)
    
    # =========================================================================
    # 图2: 改进的收敛曲线对比 (统一X轴为评估次数)
    # =========================================================================
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    nsga2_conv_df = pd.DataFrame(nsga2_history)
    random_conv_df = pd.DataFrame(random_history)
    
    # 左图: KQ收敛对比 (X轴统一为评估次数)
    ax1.plot(nsga2_conv_df['evaluations'], nsga2_conv_df['best_kq'], 
            'b-', linewidth=2, label='NSGA-II Best KQ', marker='o', markersize=4)
    ax1.plot(nsga2_conv_df['evaluations'], nsga2_conv_df['mean_kq'], 
            'b--', linewidth=1.5, label='NSGA-II Mean KQ', alpha=0.7)
    
    ax1.plot(random_conv_df['evaluations'], random_conv_df['best_kq'], 
            'r-', linewidth=2, label='Random Search Best KQ', marker='s', markersize=4)
    
    ax1.set_xlabel('Number of Evaluations', fontsize=12, fontweight='bold')
    ax1.set_ylabel('KQ (MPa·m^1/2)', fontsize=12, fontweight='bold')
    ax1.set_title('KQ Convergence Comparison', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # 右图: HTC收敛对比 (X轴统一为评估次数)
    ax2.plot(nsga2_conv_df['evaluations'], nsga2_conv_df['best_htc'], 
            'b-', linewidth=2, label='NSGA-II Best HTC', marker='o', markersize=4)
    ax2.plot(nsga2_conv_df['evaluations'], nsga2_conv_df['mean_htc'], 
            'b--', linewidth=1.5, label='NSGA-II Mean HTC', alpha=0.7)
    
    ax2.plot(random_conv_df['evaluations'], random_conv_df['best_htc'], 
            'r-', linewidth=2, label='Random Search Best HTC', marker='s', markersize=4)
    
    ax2.set_xlabel('Number of Evaluations', fontsize=12, fontweight='bold')
    ax2.set_ylabel('HTC (MPa)', fontsize=12, fontweight='bold')
    ax2.set_title('HTC Convergence Comparison', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plot2_file = os.path.join(plots_dir, f'2_convergence_comparison_{elements_str}.png')
    plt.savefig(plot2_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"已保存图2: 收敛曲线对比 (改进版-统一X轴)")
    
    convergence_data = pd.DataFrame({
        'NSGA2_Evaluations': nsga2_conv_df['evaluations'],
        'NSGA2_Best_KQ': nsga2_conv_df['best_kq'],
        'NSGA2_Mean_KQ': nsga2_conv_df['mean_kq'],
        'NSGA2_Best_HTC': nsga2_conv_df['best_htc'],
        'NSGA2_Mean_HTC': nsga2_conv_df['mean_htc'],
        'Random_Evaluations': random_conv_df['evaluations'],
        'Random_Best_KQ': random_conv_df['best_kq'],
        'Random_Best_HTC': random_conv_df['best_htc']
    })
    convergence_file = os.path.join(excel_dir, 'plot2_convergence_comparison_data.xlsx')
    convergence_data.to_excel(convergence_file, index=False)
    
    # =========================================================================
    # 图3: 组合评分收敛对比 (删除,不再生成)
    # =========================================================================
    # 这个图已被移除,因为信息价值有限
    
    # =========================================================================
    # 图4: 计算效率对比 (保持不变)
    # =========================================================================
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    methods = stats_df['Method'].tolist()
    
    evaluations = stats_df['Total_Evaluations'].tolist()
    colors = ['blue', 'red']
    bars1 = ax1.bar(methods, evaluations, color=colors, alpha=0.7, edgecolor='black')
    ax1.set_ylabel('Total Evaluations', fontsize=12, fontweight='bold')
    ax1.set_title('Total Evaluations Comparison', fontsize=14, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    times = stats_df['Time_seconds'].tolist()
    bars2 = ax2.bar(methods, times, color=colors, alpha=0.7, edgecolor='black')
    ax2.set_ylabel('Time (seconds)', fontsize=12, fontweight='bold')
    ax2.set_title('Computation Time Comparison', fontsize=14, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    
    for bar in bars2:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}s',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plot4_file = os.path.join(plots_dir, f'3_efficiency_comparison_{elements_str}.png')
    plt.savefig(plot4_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"已保存图3: 计算效率对比")
    
    efficiency_data = pd.DataFrame({
        'Method': methods,
        'Total_Evaluations': evaluations,
        'Time_seconds': times,
        'Evaluations_per_second': [e/t for e, t in zip(evaluations, times)]
    })
    efficiency_file = os.path.join(excel_dir, 'plot3_efficiency_comparison_data.xlsx')
    efficiency_data.to_excel(efficiency_file, index=False)
    
    # =========================================================================
    # 图5: 性能分布对比 (保持不变)
    # =========================================================================
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    kq_data = [nsga2_results['Predicted_KQ'].values, random_results['Predicted_KQ'].values]
    bp1 = ax1.boxplot(kq_data, labels=['NSGA-II', 'Random Search'], 
                      patch_artist=True, showmeans=True)
    
    for patch, color in zip(bp1['boxes'], ['lightblue', 'lightcoral']):
        patch.set_facecolor(color)
    
    ax1.set_ylabel('KQ (MPa·m^1/2)', fontsize=12, fontweight='bold')
    ax1.set_title('KQ Distribution Comparison', fontsize=14, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    
    htc_data = [nsga2_results['Predicted_HTC'].values, random_results['Predicted_HTC'].values]
    bp2 = ax2.boxplot(htc_data, labels=['NSGA-II', 'Random Search'], 
                      patch_artist=True, showmeans=True)
    
    for patch, color in zip(bp2['boxes'], ['lightblue', 'lightcoral']):
        patch.set_facecolor(color)
    
    ax2.set_ylabel('HTC (MPa)', fontsize=12, fontweight='bold')
    ax2.set_title('HTC Distribution Comparison', fontsize=14, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plot5_file = os.path.join(plots_dir, f'4_distribution_comparison_{elements_str}.png')
    plt.savefig(plot5_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"已保存图4: 性能分布对比")
    
    distribution_data = pd.DataFrame({
        'NSGA2_KQ': pd.Series(nsga2_results['Predicted_KQ'].values),
        'Random_KQ': pd.Series(random_results['Predicted_KQ'].values),
        'NSGA2_HTC': pd.Series(nsga2_results['Predicted_HTC'].values),
        'Random_HTC': pd.Series(random_results['Predicted_HTC'].values)
    })
    distribution_file = os.path.join(excel_dir, 'plot4_distribution_comparison_data.xlsx')
    distribution_data.to_excel(distribution_file, index=False)
    
    # =========================================================================
    # 图6: 约束满足率对比 (保持不变)
    # =========================================================================
    plt.figure(figsize=(10, 6))
    
    constraint_rates = stats_df['Constraint_Satisfaction_Rate_%'].tolist()
    colors = ['blue', 'red']
    bars = plt.bar(methods, constraint_rates, color=colors, alpha=0.7, edgecolor='black')
    
    plt.ylabel('Constraint Satisfaction Rate (%)', fontsize=12, fontweight='bold')
    plt.title('Constraint Satisfaction Rate Comparison', fontsize=14, fontweight='bold')
    plt.ylim([0, 105])
    plt.grid(axis='y', alpha=0.3)
    
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%',
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plot6_file = os.path.join(plots_dir, f'5_constraint_satisfaction_{elements_str}.png')
    plt.savefig(plot6_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"已保存图5: 约束满足率对比")
    
    constraint_data = pd.DataFrame({
        'Method': methods,
        'Constraint_Satisfaction_Rate_%': constraint_rates
    })
    constraint_file = os.path.join(excel_dir, 'plot5_constraint_satisfaction_data.xlsx')
    constraint_data.to_excel(constraint_file, index=False)
    
    # =========================================================================
    # 图7: 综合性能柱状图对比 (替代雷达图)
    # =========================================================================
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    # 提取数据
    nsga2_best_kq = stats_df[stats_df['Method'] == 'NSGA-II']['Best_KQ'].values[0]
    random_best_kq = stats_df[stats_df['Method'] == 'Random Search']['Best_KQ'].values[0]
    
    nsga2_best_htc = stats_df[stats_df['Method'] == 'NSGA-II']['Best_HTC'].values[0]
    random_best_htc = stats_df[stats_df['Method'] == 'Random Search']['Best_HTC'].values[0]
    
    nsga2_mean_kq = stats_df[stats_df['Method'] == 'NSGA-II']['Mean_KQ'].values[0]
    random_mean_kq = stats_df[stats_df['Method'] == 'Random Search']['Mean_KQ'].values[0]
    
    nsga2_mean_htc = stats_df[stats_df['Method'] == 'NSGA-II']['Mean_HTC'].values[0]
    random_mean_htc = stats_df[stats_df['Method'] == 'Random Search']['Mean_HTC'].values[0]
    
    nsga2_pareto_count = stats_df[stats_df['Method'] == 'NSGA-II']['Pareto_Solutions'].values[0]
    random_pareto_count = 0
    
    nsga2_valid_count = stats_df[stats_df['Method'] == 'NSGA-II']['Valid_Solutions'].values[0]
    random_valid_count = stats_df[stats_df['Method'] == 'Random Search']['Valid_Solutions'].values[0]
    
    # 子图1: Best KQ
    ax = axes[0]
    bars = ax.bar(methods, [nsga2_best_kq, random_best_kq], color=['blue', 'red'], alpha=0.7, edgecolor='black')
    ax.set_ylabel('KQ (MPa·m^1/2)', fontsize=11, fontweight='bold')
    ax.set_title('Best KQ Comparison', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # 子图2: Best HTC
    ax = axes[1]
    bars = ax.bar(methods, [nsga2_best_htc, random_best_htc], color=['blue', 'red'], alpha=0.7, edgecolor='black')
    ax.set_ylabel('HTC (MPa)', fontsize=11, fontweight='bold')
    ax.set_title('Best HTC Comparison', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # 子图3: Mean KQ
    ax = axes[2]
    bars = ax.bar(methods, [nsga2_mean_kq, random_mean_kq], color=['blue', 'red'], alpha=0.7, edgecolor='black')
    ax.set_ylabel('KQ (MPa·m^1/2)', fontsize=11, fontweight='bold')
    ax.set_title('Mean KQ Comparison', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # 子图4: Mean HTC
    ax = axes[3]
    bars = ax.bar(methods, [nsga2_mean_htc, random_mean_htc], color=['blue', 'red'], alpha=0.7, edgecolor='black')
    ax.set_ylabel('HTC (MPa)', fontsize=11, fontweight='bold')
    ax.set_title('Mean HTC Comparison', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # 子图5: Pareto解数量
    ax = axes[4]
    bars = ax.bar(methods, [nsga2_pareto_count, random_pareto_count], color=['blue', 'red'], alpha=0.7, edgecolor='black')
    ax.set_ylabel('Number of Solutions', fontsize=11, fontweight='bold')
    ax.set_title('Pareto Solutions Count', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # 子图6: 有效解数量
    ax = axes[5]
    bars = ax.bar(methods, [nsga2_valid_count, random_valid_count], color=['blue', 'red'], alpha=0.7, edgecolor='black')
    ax.set_ylabel('Number of Solutions', fontsize=11, fontweight='bold')
    ax.set_title('Valid Solutions Count', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.suptitle('Comprehensive Performance Comparison', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plot7_file = os.path.join(plots_dir, f'6_comprehensive_bar_comparison_{elements_str}.png')
    plt.savefig(plot7_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"已保存图6: 综合性能柱状图对比 (替代雷达图)")
    
    comprehensive_data = pd.DataFrame({
        'Metric': ['Best_KQ', 'Best_HTC', 'Mean_KQ', 'Mean_HTC', 'Pareto_Count', 'Valid_Count'],
        'NSGA-II': [nsga2_best_kq, nsga2_best_htc, nsga2_mean_kq, nsga2_mean_htc, nsga2_pareto_count, nsga2_valid_count],
        'Random_Search': [random_best_kq, random_best_htc, random_mean_kq, random_mean_htc, random_pareto_count, random_valid_count]
    })
    comprehensive_file = os.path.join(excel_dir, 'plot6_comprehensive_bar_comparison_data.xlsx')
    comprehensive_data.to_excel(comprehensive_file, index=False)
    
    print(f"\n所有图表已保存到: {plots_dir}")
    print(f"所有Excel数据已保存到: {excel_dir}")
# =============================================================================
# Cross-System Comparison
# =============================================================================

def create_cross_system_comparison(all_system_results, output_dir):
    """
    创建跨元素系统的对比分析
    """
    print("\n\n========= 创建跨元素系统对比分析 =========")
    
    cross_dir = os.path.join(output_dir, "cross_system_comparison")
    if not os.path.exists(cross_dir):
        os.makedirs(cross_dir)
    
    plots_dir = os.path.join(cross_dir, "plots")
    if not os.path.exists(plots_dir):
        os.makedirs(plots_dir)
    
    excel_dir = os.path.join(cross_dir, "excel_data")
    if not os.path.exists(excel_dir):
        os.makedirs(excel_dir)
    
    summary_data = []
    for system_data in all_system_results:
        n_elements = system_data['n_elements']
        elements_str = system_data['elements_str']
        
        for method in ['NSGA-II', 'Random Search']:
            row = {
                'System': f'{n_elements}-Element',
                'Elements': elements_str,
                'Method': method
            }
            
            stats = system_data['stats_df']
            method_row = stats[stats['Method'] == method].iloc[0]
            
            row['Best_KQ'] = method_row['Best_KQ']
            row['Best_HTC'] = method_row['Best_HTC']
            row['Mean_KQ'] = method_row['Mean_KQ']
            row['Mean_HTC'] = method_row['Mean_HTC']
            row['Time_seconds'] = method_row['Time_seconds']
            row['Total_Evaluations'] = method_row['Total_Evaluations']
            row['Constraint_Rate_%'] = method_row['Constraint_Satisfaction_Rate_%']
            row['Best_Combined_Score'] = method_row['Best_Combined_Score']
            
            summary_data.append(row)
    
    summary_df = pd.DataFrame(summary_data)
    summary_file = os.path.join(excel_dir, 'cross_system_summary.xlsx')
    summary_df.to_excel(summary_file, index=False)
    print(f"已保存跨系统汇总数据: {summary_file}")
    
    plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    systems = sorted(summary_df['System'].unique(), key=lambda x: int(x.split('-')[0]))
    nsga2_data = summary_df[summary_df['Method'] == 'NSGA-II']
    random_data = summary_df[summary_df['Method'] == 'Random Search']
    
    x_pos = np.arange(len(systems))
    width = 0.35
    
    nsga2_best_kq = [nsga2_data[nsga2_data['System'] == s]['Best_KQ'].values[0] for s in systems]
    random_best_kq = [random_data[random_data['System'] == s]['Best_KQ'].values[0] for s in systems]
    
    ax1.bar(x_pos - width/2, nsga2_best_kq, width, label='NSGA-II', color='blue', alpha=0.7)
    ax1.bar(x_pos + width/2, random_best_kq, width, label='Random Search', color='red', alpha=0.7)
    ax1.set_xlabel('Alloy System', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Best KQ (MPa·m^1/2)', fontsize=12, fontweight='bold')
    ax1.set_title('Best KQ Across Systems', fontsize=14, fontweight='bold')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(systems)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    nsga2_best_htc = [nsga2_data[nsga2_data['System'] == s]['Best_HTC'].values[0] for s in systems]
    random_best_htc = [random_data[random_data['System'] == s]['Best_HTC'].values[0] for s in systems]
    
    ax2.bar(x_pos - width/2, nsga2_best_htc, width, label='NSGA-II', color='blue', alpha=0.7)
    ax2.bar(x_pos + width/2, random_best_htc, width, label='Random Search', color='red', alpha=0.7)
    ax2.set_xlabel('Alloy System', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Best HTC (MPa)', fontsize=12, fontweight='bold')
    ax2.set_title('Best HTC Across Systems', fontsize=14, fontweight='bold')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(systems)
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    nsga2_time = [nsga2_data[nsga2_data['System'] == s]['Time_seconds'].values[0] for s in systems]
    random_time = [random_data[random_data['System'] == s]['Time_seconds'].values[0] for s in systems]
    
    ax3.bar(x_pos - width/2, nsga2_time, width, label='NSGA-II', color='blue', alpha=0.7)
    ax3.bar(x_pos + width/2, random_time, width, label='Random Search', color='red', alpha=0.7)
    ax3.set_xlabel('Alloy System', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Time (seconds)', fontsize=12, fontweight='bold')
    ax3.set_title('Computation Time Across Systems', fontsize=14, fontweight='bold')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(systems)
    ax3.legend()
    ax3.grid(axis='y', alpha=0.3)
    
    nsga2_constraint = [nsga2_data[nsga2_data['System'] == s]['Constraint_Rate_%'].values[0] for s in systems]
    random_constraint = [random_data[random_data['System'] == s]['Constraint_Rate_%'].values[0] for s in systems]
    
    ax4.bar(x_pos - width/2, nsga2_constraint, width, label='NSGA-II', color='blue', alpha=0.7)
    ax4.bar(x_pos + width/2, random_constraint, width, label='Random Search', color='red', alpha=0.7)
    ax4.set_xlabel('Alloy System', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Constraint Satisfaction Rate (%)', fontsize=12, fontweight='bold')
    ax4.set_title('Constraint Satisfaction Across Systems', fontsize=14, fontweight='bold')
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels(systems)
    ax4.set_ylim([0, 105])
    ax4.legend()
    ax4.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plot_file = os.path.join(plots_dir, 'cross_system_comparison.png')
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"已保存跨系统对比图: {plot_file}")
    
    cross_comparison_data = pd.DataFrame({
        'System': systems + systems,
        'Method': ['NSGA-II'] * len(systems) + ['Random Search'] * len(systems),
        'Best_KQ': nsga2_best_kq + random_best_kq,
        'Best_HTC': nsga2_best_htc + random_best_htc,
        'Time_seconds': nsga2_time + random_time,
        'Constraint_Rate_%': nsga2_constraint + random_constraint
    })
    cross_data_file = os.path.join(excel_dir, 'cross_system_comparison_data.xlsx')
    cross_comparison_data.to_excel(cross_data_file, index=False)
    
    print(f"\n跨系统对比分析完成！")
    print(f"结果保存在: {cross_dir}")

# =============================================================================
# Main Function
# =============================================================================

def main():
    base_dir = r"D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.20-回复审稿人2-C2\NSGA-II_vs_RandomSearch_Comparison-1.28"
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = os.path.join(base_dir, f"comparison_{timestamp}")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    print("======= NSGA-II vs Random Search 对比分析程序 =======")
    print("本程序将对比NSGA-II和随机搜索在合金优化中的性能")
    print("对比系统: 5元、6元、7元、8元合金")
    print(f"所有结果将保存至: {output_dir}")
    
    print("\n加载KQ模型和缩放器...")
    kq_model_dir = r"D:\文成数据库\训练模型3.4-KQ"
    kq_model_path = os.path.join(kq_model_dir, 'KQ_model.joblib')
    kq_scaler_path = os.path.join(kq_model_dir, 'final_scaler.joblib')
    
    print("加载HTC模型和缩放器...")
    htc_model_dir = r"D:\文成数据库\训练模型高温压缩-10.9"
    htc_model_path = os.path.join(htc_model_dir, 'high_temperature_compression_model.joblib')
    htc_scaler_path = os.path.join(htc_model_dir, 'final_scaler.joblib')
    
    for path in [kq_model_path, kq_scaler_path, htc_model_path, htc_scaler_path]:
        if not os.path.exists(path):
            print(f"错误: 文件 {path} 不存在!")
            return
    
    try:
        kq_model = joblib.load(kq_model_path)
        kq_scaler = joblib.load(kq_scaler_path)
        htc_model = joblib.load(htc_model_path)
        htc_scaler = joblib.load(htc_scaler_path)
        print("模型和缩放器加载成功.")
    except Exception as e:
        print(f"加载模型或缩放器时出错: {e}")
        return
    
    element_combinations = [
        ['Nb', 'Si', 'Ti', 'Al', 'Cr'],
        ['Nb', 'Si', 'Ti', 'Al', 'Cr', 'Hf'],
        ['Nb', 'Si', 'Ti', 'Al', 'Cr', 'Hf', 'Zr'],
        ['Nb', 'Si', 'Ti', 'Al', 'Cr', 'Hf', 'Zr', 'V']
    ]
    
    all_system_results = []
    
    for idx, elements in enumerate(element_combinations):
        num_elements = len(elements)
        elements_str = ''.join([el[0] for el in elements])
        
        print(f"\n\n{'='*80}")
        print(f"========= 开始{num_elements}元合金对比分析 ({', '.join(elements)}) =========")
        print(f"{'='*80}")
        
        print("\n========= 第一步: 运行NSGA-II =========")
        nsga2_optimizer = CustomNSGA2(
            kq_model=kq_model,
            kq_scaler=kq_scaler,
            htc_model=htc_model,
            htc_scaler=htc_scaler,
            elements_list=elements,
            pop_size=500,
            max_generations=250,
            kq_weight=0.75,
            htc_weight=0.25,
            min_htc_threshold=30,
            random_seed=42
        )
        
        nsga2_results, nsga2_time, nsga2_history = nsga2_optimizer.run(verbose=True)
        nsga2_total_evals = nsga2_optimizer.total_evaluations
        
        if nsga2_results is None:
            print(f"{num_elements}元合金NSGA-II未找到有效解决方案，跳过该系统。")
            continue
        
        print(f"\nNSGA-II完成！")
        print(f"- 运行时间: {nsga2_time:.2f}秒")
        print(f"- 总评估次数: {nsga2_total_evals}")
        print(f"- 找到有效解: {len(nsga2_results)}个")
        print(f"- Pareto解: {len(nsga2_results[nsga2_results['Is_Pareto'] == True])}个")
        print(f"- 最佳KQ: {nsga2_results['Predicted_KQ'].max():.2f}")
        print(f"- 最佳HTC: {nsga2_results['Predicted_HTC'].max():.2f}")
        
        print("\n========= 第二步: 运行随机搜索 =========")
        random_optimizer = RandomSearch(
            kq_model=kq_model,
            kq_scaler=kq_scaler,
            htc_model=htc_model,
            htc_scaler=htc_scaler,
            elements_list=elements,
            n_samples=10000,
            kq_weight=0.75,
            htc_weight=0.25,
            min_htc_threshold=30,
            random_seed=42
        )
        
        random_results, random_time, random_constraint_rate, random_history = random_optimizer.run(verbose=True)
        random_total_evals = random_optimizer.total_evaluations
        
        if random_results is None:
            print(f"{num_elements}元合金随机搜索未找到有效解决方案，跳过该系统。")
            continue
        
        print(f"\n随机搜索完成！")
        print(f"- 运行时间: {random_time:.2f}秒")
        print(f"- 总评估次数: {random_total_evals}")
        print(f"- 找到有效解: {len(random_results)}个")
        print(f"- 约束满足率: {random_constraint_rate:.2f}%")
        print(f"- 最佳KQ: {random_results['Predicted_KQ'].max():.2f}")
        print(f"- 最佳HTC: {random_results['Predicted_HTC'].max():.2f}")
        
        print("\n========= 第三步: 对比分析与可视化 =========")
        stats_df, comparison_dir = compare_methods(
            nsga2_results, nsga2_time, nsga2_history, nsga2_total_evals,
            random_results, random_time, random_constraint_rate, random_history, random_total_evals,
            elements, output_dir
        )
        
        print("\n========= 对比分析总结 =========")
        print("\n统计指标对比:")
        print(stats_df.to_string(index=False))
        
        all_system_results.append({
            'n_elements': num_elements,
            'elements_str': elements_str,
            'elements_list': elements,
            'stats_df': stats_df,
            'comparison_dir': comparison_dir
        })
    
    if len(all_system_results) > 1:
        create_cross_system_comparison(all_system_results, output_dir)
    
    print(f"\n\n{'='*80}")
    print("所有系统对比分析完成！")
    print(f"{'='*80}")
    print(f"\n主目录: {output_dir}")
    print("\n包含以下内容:")
    print("  1. comparison_5元_NSTAC/  - 5元合金对比结果")
    print("  2. comparison_6元_NSTACHf/ - 6元合金对比结果")
    print("  3. comparison_7元_NSTACHfZr/ - 7元合金对比结果")
    print("  4. comparison_8元_NSTACHfZrV/ - 8元合金对比结果")
    print("  5. cross_system_comparison/ - 跨系统对比分析")
    
    print("\n每个系统包含:")
    print("  - plots/        - 6张对比可视化图表（PNG格式）")
    print("  - excel_data/   - 所有图表的原始数据（Excel格式）")
    
    print("\n跨系统对比包含:")
    print("  - plots/        - 跨系统性能对比图")
    print("  - excel_data/   - 跨系统汇总数据")
    
    print("\n图表改进说明:")
    print("  - 图2: 收敛曲线已改进为统一X轴(评估次数)便于直接对比")
    print("  - 图3: 原组合评分收敛图已删除(信息价值有限)")
    print("  - 图7: 雷达图已替换为6子图柱状图(更直观易读)")
    
    print("\n程序运行完成！")

if __name__ == "__main__":
    main()

======= NSGA-II vs Random Search 对比分析程序 =======
本程序将对比NSGA-II和随机搜索在合金优化中的性能
对比系统: 5元、6元、7元、8元合金
所有结果将保存至: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.20-回复审稿人2-C2\NSGA-II_vs_RandomSearch_Comparison-1.28\comparison_20260128_234701

加载KQ模型和缩放器...
加载HTC模型和缩放器...
模型和缩放器加载成功.


========= 开始5元合金对比分析 (Nb, Si, Ti, Al, Cr) =========

========= 第一步: 运行NSGA-II =========

运行NSGA-II优化 (Nb, Si, Ti, Al, Cr)...
种群大小: 500, 最大代数: 250
代数 50/250
代数 100/250
代数 150/250
代数 200/250
代数 250/250

NSGA-II完成！
- 运行时间: 127.67秒
- 总评估次数: 125000
- 找到有效解: 124558个
- Pareto解: 22625个
- 最佳KQ: 15.37
- 最佳HTC: 678.53

========= 第二步: 运行随机搜索 =========

运行随机搜索优化 (Nb, Si, Ti, Al, Cr)...
采样次数: 10000
采样 2000/10000
采样 4000/10000
采样 6000/10000
采样 8000/10000
采样 10000/10000

随机搜索完成！
- 运行时间: 3.58秒
- 总评估次数: 10000
- 找到有效解: 2672个
- 约束满足率: 66.99%
- 最佳KQ: 15.34
- 最佳HTC: 636.71

========= 第三步: 对比分析与可视化 =========

========= 开始对比分析 =========
已保存统计对比数据: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.20-回复审稿人2-C2\NSGA-II_vs_RandomSearch_C